# Run MODFLOW-96 with FloPy

Local FloPy workflow (no Tapis calls):
1. Load an existing model from shared model storage
2. Point it to a local tutorial output workspace
3. Write and run model input
4. Check expected output files

This notebook is a compact local FloPy workflow for running an existing MODFLOW-96 model. It is intentionally shorter than the Gulf model notebook, but the same pattern applies: configure paths, prepare the model workspace, run the model, and verify outputs before interpretation.

## Before You Run

- Make sure the model input files are available from the configured shared storage or CKAN source.
- Make sure the correct MODFLOW executable is available for this model version.
- Review the path variables before running cells that stage files or write outputs.

## Expected Outputs

- A local tutorial run workspace is created under `model_output_directory/`.
- The existing model is loaded or staged for inspection with FloPy.
- The model run reports whether it succeeded.
- Expected model output files are checked before moving on.


## Imports And Path Setup

This cell imports FloPy and supporting utilities, then sets the local paths used by the run. Review the model source, output directory, and executable settings before continuing.


In [ ]:
from pathlib import Path
from shutil import which
import os

import flopy

from modflow_utils import download_ckan_dataset


## Download Or Stage Model Inputs

The notebook downloads or locates the model inputs, then prepares a local tutorial workspace. This keeps original model files separate from generated run output.


In [ ]:
# Paths and executable configuration
local_modeldir = Path(r"/corral-repl/tacc/aci/PT2050/projects/PTDATAX-272/workingGAMs/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate")
modeldir = None  # set below based on USE_CKAN_DATA
run_dir = Path(r"model_output_directory/modflow_96")
exe_name = r"mf96"

# Optional CKAN dataset download
# Set USE_CKAN_DATA=True and provide CKAN_DATASET_URL to stage resources into ./data
USE_CKAN_DATA = False
CKAN_DATASET_URL = "https://ckan.tacc.utexas.edu/dataset/REPLACE_WITH_DATASET_SLUG"
CKAN_DATA_ROOT = Path("./data")
CKAN_MODEL_SUBDIR = ""  # Optional path under downloaded dataset directory
CKAN_EXTRACT_ZIPS = True
CKAN_OVERWRITE = False
CKAN_MAX_RESOURCES = None  # Optional int limit for first N resources

if USE_CKAN_DATA:
    if "REPLACE_WITH_DATASET_SLUG" in CKAN_DATASET_URL:
        raise ValueError("Set CKAN_DATASET_URL to a real dataset page URL before USE_CKAN_DATA=True")

    staged_dataset_dir = download_ckan_dataset(
        CKAN_DATASET_URL,
        CKAN_DATA_ROOT,
        extract_zips=CKAN_EXTRACT_ZIPS,
        overwrite=CKAN_OVERWRITE,
        max_resources=CKAN_MAX_RESOURCES,
    )

    modeldir = staged_dataset_dir / CKAN_MODEL_SUBDIR if CKAN_MODEL_SUBDIR else staged_dataset_dir
    print(f"Using CKAN-staged modeldir: {modeldir}")
else:
    modeldir = local_modeldir
    print(f"Using local modeldir: {modeldir}")

# Auto-install executable with FloPy utility if missing on PATH
if which(exe_name) is None:
    bindir = Path("/tmp/bin")
    bindir.mkdir(parents=True, exist_ok=True)
    os.environ["PATH"] = f"{bindir}{os.pathsep}{os.environ.get('PATH', '')}"

    print(f"Attempting to install '{exe_name}' into {bindir} using flopy.utils.get_modflow...")
    try:
        import inspect

        gm_kwargs = {}
        gm_sig = inspect.signature(flopy.utils.get_modflow)
        if "subset" in gm_sig.parameters:
            gm_kwargs["subset"] = [exe_name]

        flopy.utils.get_modflow(str(bindir), **gm_kwargs)
    except Exception as err:
        print(f"Auto-install attempt with subset failed: {err}")
        print("Retrying full executable bundle install...")
        try:
            flopy.utils.get_modflow(str(bindir))
        except Exception as err2:
            print(f"Auto-install failed: {err2}")

print(f"FloPy version: {flopy.__version__}")
print(f"CKAN dataset URL (optional): {CKAN_DATASET_URL}")
print(f"Source model directory: {modeldir}")
print(f"Run workspace: {run_dir}")
print(f"Executable: {exe_name}")

if not modeldir.exists():
    raise FileNotFoundError(f"Model directory not found: {modeldir}")

exe_path = which(exe_name)
if exe_path is None:
    print(f"WARNING: executable '{exe_name}' is not on PATH.")
    print("Set exe_name to a full path if needed before running model.")
else:
    print(f"Found executable: {exe_path}")

run_dir.mkdir(parents=True, exist_ok=True)



## Load The Existing Model

FloPy loads the existing MODFLOW-96 model so students can see how an older model version is handled from Python.


In [ ]:
preferred_namefiles = ["trnt_h_ss.nam", "model.nam"]
namefile = next((n for n in preferred_namefiles if (modeldir / n).exists()), None)
if namefile is None:
    nam_candidates = sorted(modeldir.glob("*.nam"))
    if not nam_candidates:
        raise FileNotFoundError(f"No MODFLOW-96 .nam file found in {modeldir}")
    namefile = nam_candidates[0].name

print(f"Using name file: {namefile}")

model_obj = flopy.modflow.Modflow.load(
    f=namefile,
    version="mf2k",
    exe_name=exe_name,
    model_ws=str(modeldir),
    check=False,
    verbose=True,
)
model_obj.change_model_ws(str(run_dir), reset_external=True)


## Write Model Inputs

This cell writes model input files into the tutorial workspace when supported by the loaded model object. Writing the files before running keeps generated outputs separate from the original source model.


In [ ]:
# Write model files into local run workspace
write_method = getattr(model_obj, "write_simulation", None)
if callable(write_method):
    write_method()
else:
    model_obj.write_input()


## Run The Model

This cell launches MODFLOW-96 through FloPy and reports whether the run succeeded. Stop here if the model does not terminate normally.


In [ ]:
# Run model through FloPy
run_method = getattr(model_obj, "run_simulation", None)
if callable(run_method):
    success, buff = run_method()
else:
    success, buff = model_obj.run_model(silent=False, report=True)

print(f"Success: {success}")
if not success:
    print("Model did not terminate normally.")


## Check Expected Outputs

This cell confirms that expected output files were created before students interpret the run.


In [ ]:
# Output checks
list_candidates = sorted(run_dir.glob("*.lst"))
for f in list_candidates[:5]:
    print(f"List file: {f.name}")

if not list_candidates:
    print("No .lst file found yet in run workspace.")
